In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [2]:
class TreeNode: 
    def __init__(self , feature_index = None , threshold = None , left = None , right = None , * , value = None): 
        self.feature_index = feature_index # index of feature to split on
        self.threshold = threshold # value of threshold(based what value we splitted)
        self.left = left 
        self.right = right
        self.value = value
    def is_leaf(self): 
        return self.value is not None 

In [11]:
class DecisionTreeRegressor: 
    def __init__(self , max_depth = 5 , min_samples_split = 2): 
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None 

    def fit(self , X , y): 
        self.root = self._build_tree(X , y , curr_depth = 0)
    def _build_tree(self , X , y , curr_depth = 0):
        n_samples  = X.shape[0]
        n_labels = len(np.unique(y))
        # stopping conditions 
        if curr_depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1: 
            # make the curr node as leaf 
            leaf_value = np.mean(y)
            return TreeNode(value = leaf_value)

        # try to split and find best feature and threshold 
        feature_index , threshold = self._best_split(X , y)

        # if no split possible 
        if feature_index is None:
            # make the curr node as leaf 
            leaf_value = np.mean(y)
            return TreeNode(value = leaf_value)

        # split based on feature index 
        left_mask = X[ : , feature_index] <= threshold
        right_mask = ~left_mask

        left_child = self._build_tree(X[left_mask] , y[left_mask] , curr_depth + 1)
        right_child = self._build_tree(X[right_mask] , y[right_mask] , curr_depth + 1)

        newNode = TreeNode(feature_index = feature_index , threshold = threshold , left = left_child , right = right_child)
        return newNode
    def _best_split(self , X , y): 
        best_mse = float('inf')
        best_idx , best_threshold = None , None

        n_samples , n_features = X.shape

        for feature_index in range(n_features): 
            thresholds = np.unique(X[ : , feature_index])
            for threshold in thresholds: 
                left_mask = X[ : , feature_index] <= threshold
                right_mask = ~left_mask

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0: 
                    continue

                mse_left = self._mse(y[left_mask]) 
                mse_right = self._mse(y[right_mask])

                weighted_mse = ((np.sum(left_mask) * mse_left) + (np.sum(right_mask) * mse_right)) / len(y) 

                if weighted_mse < best_mse: # curr_variance reduction higher  
                   best_mse = weighted_mse
                   best_idx = feature_index
                   best_threshold = threshold

        return best_idx , best_threshold
    def _mse(self, y):
        if len(y) == 0:
            return 0
        mean = np.mean(y)
        return np.mean((y - mean) ** 2) 

    def predict(self , X): 
        return np.array([self._predict_row(x , self.root) for x in X])

    def _predict_row(self , row , node):
        if node.value is not None: # means leaf node 
            return node.value

        if row[node.feature_index] <= node.threshold: 
            return self._predict_row(row , node.left)
        else:
            return self._predict_row(row , node.right)

In [14]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error , r2_score

In [5]:
X, y = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [12]:
tree = DecisionTreeRegressor(max_depth = 4)
tree.fit(X_train, y_train)

In [13]:
y_pred = tree.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

MSE: 336.5908058402483


In [15]:
r2_score(y_test , y_pred)

0.943956873337849